# Source E: San Bernardino + Riverside Medi-Cal FFS providers

This notebook turns the statewide Source E provider listing into a cleaned, county-filtered **location-level** dataset for San Bernardino and Riverside counties. Run the cells from top to bottom. It prints a compact set of results and writes a CSV for downstream analysis.

**Counting rule:** one row is a provider service-location / specialty record. A provider with multiple addresses, locations, or specialties is intentionally allowed to appear multiple times. Only byte-for-byte-equivalent rows after basic whitespace cleanup are removed. Provider counts are reported separately using provider identifiers. This is an enrolled FFS listing, not a measure of active appointment capacity or patient access.

## 1. Find and load both statewide CSV parts

The CSV columns are loaded as text to protect identifiers, ZIP codes, license numbers, and phone numbers from numeric conversion. Blank cells become missing; literal source values such as `N/A` are retained.

In [ ]:
from pathlib import Path
import re
import pandas as pd
from IPython.display import display

relative_partition_dir = Path(
    'data/provider-side/E-mediCal-fee-for-service-enrolled-providers/'
    'zipped_bc_too_big_for_github'
)
repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / relative_partition_dir).is_dir()),
    None,
)
if repo_root is None:
    raise FileNotFoundError(
        f'Cannot find {relative_partition_dir} from {Path.cwd()}. '
        'Start Jupyter in this repository or one of its subfolders.'
    )

partition_dir = repo_root / relative_partition_dir
csv_files = [
    partition_dir / 'medi-cal-ffs-provider-listing_10_26_26_PART 1 first 200k line.csv',
    partition_dir / 'medi-cal-ffs-provider-listing_10_26_26_PART 2 next 180k line.csv',
]
missing_files = [str(path) for path in csv_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError('Missing Source E CSV file(s):\n' + '\n'.join(missing_files))

providers_raw = pd.concat(
    [pd.read_csv(path, encoding='cp1252', dtype='string', keep_default_na=False) for path in csv_files],
    ignore_index=True,
)
print(f'Loaded {len(providers_raw):,} statewide rows and {len(providers_raw.columns)} columns.')
print('Input files:', *(path.name for path in csv_files), sep='\n  - ')

## 2. Clean text and retain meaningful repeated records

Trim surrounding whitespace and convert empty strings to missing values. Keep source fields and literal placeholders unchanged otherwise. IDs remain strings; values accidentally parsed earlier with a trailing `.0` are normalized. The exact-row deduplication does **not** collapse different addresses, service locations, specialties, enrollment dates, or provider records.

In [ ]:
providers_df = providers_raw.copy()
for column in providers_df.select_dtypes(include=['string', 'object']).columns:
    providers_df[column] = providers_df[column].str.strip().replace('', pd.NA)

# Normalize only identifier fields; never cast them to numbers (leading zeros matter).
identifier_columns = [c for c in ['Provider_Number', 'NPI', 'Owner_Number', 'Service_Location_Number'] if c in providers_df]
for column in identifier_columns:
    providers_df[column] = providers_df[column].str.replace(r'^(\d+)\.0$', r'\1', regex=True)

rows_before_dedup = len(providers_df)
providers_df = providers_df.drop_duplicates().copy()
exact_duplicates_removed = rows_before_dedup - len(providers_df)

target_counties = {'SAN BERNARDINO', 'RIVERSIDE'}
def county_key(series):
    return (
        series.astype('string').str.strip().str.upper()
        .str.replace(r'\s+COUNTY$', '', regex=True)
    )

# County is the source administrative field; CountyName is coordinate-derived.
county_source = county_key(providers_df['County']) if 'County' in providers_df else pd.Series(pd.NA, index=providers_df.index, dtype='string')
county_geo = county_key(providers_df['CountyName']) if 'CountyName' in providers_df else pd.Series(pd.NA, index=providers_df.index, dtype='string')
in_target = county_source.isin(target_counties) | county_geo.isin(target_counties)
county_providers_df = providers_df.loc[in_target].copy()
county_providers_df['County_Analysis'] = county_source.loc[in_target].where(
    county_source.loc[in_target].isin(target_counties), county_geo.loc[in_target]
)
county_providers_df['County_Analysis'] = county_providers_df['County_Analysis'].str.title()
county_providers_df['County_Source_Mismatch'] = (
    county_source.loc[in_target].isin(target_counties)
    & county_geo.loc[in_target].isin(target_counties)
    & county_source.loc[in_target].ne(county_geo.loc[in_target])
).to_numpy()

# Useful standardized location fields; original Address/ZIP/coordinates remain intact.
if {'Address', 'Address2'}.issubset(county_providers_df.columns):
    county_providers_df['Full_Address'] = (
        county_providers_df[['Address', 'Address2']].fillna('').agg(' '.join, axis=1)
        .str.replace(r'\s+', ' ', regex=True).str.strip().replace('', pd.NA)
    )
if 'ZIP' in county_providers_df.columns:
    county_providers_df['ZIP5'] = county_providers_df['ZIP'].astype('string').str.extract(r'(\d{5})', expand=False)
for coordinate in ['Latitude', 'Longitude']:
    if coordinate in county_providers_df.columns:
        county_providers_df[coordinate] = pd.to_numeric(county_providers_df[coordinate], errors='coerce')
if 'Enroll_Status_Eff_DT' in county_providers_df.columns:
    county_providers_df['Enroll_Status_Eff_DT_Parsed'] = pd.to_datetime(
        county_providers_df['Enroll_Status_Eff_DT'], errors='coerce'
    )

print(f'Exact duplicate rows removed: {exact_duplicates_removed:,}')
print(f'Target-county location/specialty rows retained: {len(county_providers_df):,}')
print('County assignment uses County when it matches the target list; otherwise coordinate-derived CountyName.')

## 3. Results: county and provider-type summaries

The primary output remains one row per distinct source record/location/specialty. Provider counts below use NPI when present and fall back to Provider_Number; they are not added across counties as if every location were a unique provider. `FI_Provider_Type` and specialty fields are the program's provider classification, while `ANC_Provider_Type` and `Provider_Taxonomy` are retained as additional classifications.

In [ ]:
if county_providers_df.empty:
    print('No rows matched San Bernardino or Riverside County. Check the source county fields.')
else:
    provider_id = county_providers_df.get('NPI', pd.Series(pd.NA, index=county_providers_df.index, dtype='string'))
    fallback_id = county_providers_df.get('Provider_Number', pd.Series(pd.NA, index=county_providers_df.index, dtype='string'))
    county_providers_df['Provider_Key'] = provider_id.where(provider_id.notna() & provider_id.ne(''), fallback_id)
    county_providers_df['Provider_Key'] = county_providers_df['Provider_Key'].fillna(
        'ROW-' + county_providers_df.index.astype(str)
    )

    county_summary = (county_providers_df.groupby('County_Analysis', dropna=False)
        .agg(location_specialty_rows=('Provider_Key', 'size'),
             unique_providers=('Provider_Key', 'nunique'),
             cities=('City', 'nunique') if 'City' in county_providers_df else ('Provider_Key', 'size'),
             ZIP5_areas=('ZIP5', 'nunique') if 'ZIP5' in county_providers_df else ('Provider_Key', 'size'))
        .sort_index())
    print('County-level counts:')
    display(county_summary)

    type_columns = [c for c in ['County_Analysis', 'FI_Provider_Type_CD', 'FI_Provider_Type'] if c in county_providers_df.columns]
    type_summary = (county_providers_df.groupby(type_columns, dropna=False)
        .agg(location_specialty_rows=('Provider_Key', 'size'),
             unique_providers=('Provider_Key', 'nunique'))
        .reset_index().sort_values(['County_Analysis', 'location_specialty_rows'], ascending=[True, False]))
    print('Provider types by county (top 20 per county):')
    display(type_summary.groupby('County_Analysis', group_keys=False).head(20))

    specialty_columns = [c for c in ['County_Analysis', 'FI_Provider_Specialty_CD', 'FI_Provider_Specialty'] if c in county_providers_df.columns]
    specialty_summary = (county_providers_df.groupby(specialty_columns, dropna=False)
        .agg(location_specialty_rows=('Provider_Key', 'size'),
             unique_providers=('Provider_Key', 'nunique'))
        .reset_index().sort_values(['County_Analysis', 'location_specialty_rows'], ascending=[True, False]))
    print('Specialties by county (top 20 per county):')
    display(specialty_summary.groupby('County_Analysis', group_keys=False).head(20))

    print('Location-level records (preview):')
    display(county_providers_df.head(10))

## 4. Export the cleaned county-level records

The CSV preserves all source attributes plus analysis fields. No provider is collapsed to a single address.

In [ ]:
output_dir = repo_root / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'source_e_san_bernardino_riverside_ffs_providers.csv'
county_providers_df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f'Wrote {len(county_providers_df):,} location/specialty rows to: {output_path}')
print(f'Output columns ({len(county_providers_df.columns)}): {county_providers_df.columns.tolist()}')

## Interpretation and caveats

- `County_Analysis` is the county label used to filter and group the data. Both the administrative `County` field and coordinate-derived `CountyName` are checked so that missing values in one do not automatically discard an otherwise county-matched location. Review `County_Source_Mismatch` rows before relying on county totals.
- Counts represent enrolled FFS listing records at the source's point-in-time snapshot. They do not establish that a provider is accepting patients, available, in-network for a particular managed-care plan, or currently practicing at the listed address.
- A single NPI can have many valid addresses, specialties, service-location numbers, and enrollment records. Use the location-level CSV for geography; use `unique_providers` for a deduplicated provider-identifier count.